In [1]:
import sys
import os

# Ativa o recarregamento automático
%load_ext autoreload
%autoreload 2

# Define os caminhos absolutos necessários
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) # D:\GeoPipe
pasta_src = os.path.join(raiz_projeto, "src")                         # D:\GeoPipe\src

# Adiciona ambos ao sys.path se já não estiverem lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)
if pasta_src not in sys.path:
    sys.path.append(pasta_src)

# Agora o import vai funcionar e o 'nodes.py' vai encontrar a pasta 'utils'
from src.fmask_pipeline.pipelines.cloud_preprocess.nodes import cloud_removal


In [2]:
import pandas as pd
import os

## Copy images

In [3]:
temp_image_dir = "../data/temp/interpolation/images/"
temp_mask_dir = "../data/temp/interpolation/masks/"
temp_clean_image_dir = "../data/temp/interpolation/clean_images/"
temp_color_log_dir = "../data/temp/interpolation/color_logs/"

In [4]:
locations = [location for location in os.listdir("../../data/02_boa_images") if '.' not in location]

In [5]:
locations

['argemiro', 'engenheiro_avidos', 'gramame', 'lagoa_do_arroz', 'mares', 'sume']

In [6]:
for location in locations:
    os.makedirs(temp_image_dir + "/" + location, exist_ok=True)
    os.makedirs(temp_mask_dir + "/" + location + "/fmask", exist_ok=True)
    os.makedirs(temp_clean_image_dir + "/" + location  + "/fmask" + "/temporal_interpolation", exist_ok=True)
    os.makedirs(temp_color_log_dir + "/" + location, exist_ok=True)

In [7]:
images = pd.read_csv("../data/temporal_interpolation_images.csv")

In [8]:
images.columns

Index(['mask', 'cloud_percentage', 'cloud_shadow_percentage', 'reservoir',
       'image_path', 'random_mask', 'random_cloud_percentage',
       'random_cloud_shadow_percentage'],
      dtype='object')

In [9]:
images.iloc[0, 4], images.iloc[0, 5]

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2025\\mask_sentinel_TOA_S2_argemiro_20251209.tif')

In [10]:
images_path = images["image_path"].tolist()
masks_path = images["random_mask"].tolist()

In [11]:
images_path[0], masks_path[0] 

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2025\\mask_sentinel_TOA_S2_argemiro_20251209.tif')

In [12]:
all_dates = [image.split('_')[-1].split('.')[0] for image in images_path]
all_dates = sorted(list(set(all_dates)))

init_date = all_dates[0]
final_date = all_dates[-1]

# format data to yyyy-mm-dd
init_date_formatted = f"{init_date[:4]}-{init_date[4:6]}-{init_date[6:]}"
final_date_formatted = f"{final_date[:4]}-{final_date[4:6]}-{final_date[6:]}"

print(f"Initial date: {init_date_formatted}")
print(f"Final date: {final_date_formatted}")

Initial date: 2017-05-16
Final date: 2026-07-30


In [13]:
for location in locations:
    for year in range(int(init_date[:4]), int(final_date[:4]) + 1):
        os.makedirs(os.path.join(temp_image_dir, location, str(year)), exist_ok=True)
        os.makedirs(os.path.join(temp_mask_dir, location, "fmask", str(year)), exist_ok=True)
        os.makedirs(os.path.join(temp_clean_image_dir, location, "fmask", "temporal_interpolation", str(year)), exist_ok=True)
        os.makedirs(os.path.join(temp_color_log_dir, location, str(year)), exist_ok=True)

In [14]:
import os
import shutil

# Otimização 1: Criar conjuntos de arquivos já existentes para busca em tempo O(1)
# Substitui o lento 'os.path.exists' dentro do loop externo
exist_images = set()
exist_masks = set()

for location in locations:
    # Otimização 2: Pré-criar as pastas base para evitar chamadas redundantes
    os.makedirs(os.path.join(temp_image_dir, location), exist_ok=True)
    os.makedirs(os.path.join(temp_mask_dir, location, 'fmask'), exist_ok=True)

    for image_path, mask_path in zip(images_path, masks_path):
        if location not in image_path or location not in mask_path:
            continue
        
        print(f"Processing image: {image_path} and mask: {mask_path}, location: {location}")
        # Otimização 3: Extração de strings simplificada e direta
        image_filename = os.path.basename(image_path)
        
        # Extrai o final do arquivo (ex: "2026_abc.png")
        img_suffix = image_filename.split('_')[-1]
        year = img_suffix[:4]
        
        # Reconstrói o nome da máscara de forma limpa
        mask_filename = f"fake_{os.path.basename(mask_path)}"
        mask_filename = f"{mask_filename.rsplit('_', 1)[0]}_{img_suffix}"
        
        # Define os caminhos finais
        dest_image_dir = os.path.join(temp_image_dir, location, year)
        dest_mask_dir = os.path.join(temp_mask_dir, location, 'fmask', year)
        
        temp_image_path = os.path.join(dest_image_dir, image_filename)
        temp_mask_path = os.path.join(dest_mask_dir, mask_filename)

        # Otimização 4: Cria a subpasta específica do ano apenas se necessário
        os.makedirs(dest_image_dir, exist_ok=True)
        os.makedirs(dest_mask_dir, exist_ok=True)

        # Otimização 5: Copia apenas se o arquivo de destino final não existir
        if not os.path.exists(temp_image_path):
            shutil.copy2(image_path, temp_image_path)
            
        if not os.path.exists(temp_mask_path):
            shutil.copy2(mask_path, temp_mask_path)


Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190607.tif and mask: D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\2025\mask_sentinel_TOA_S2_argemiro_20251209.tif, location: argemiro


Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190717.tif and mask: D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\2019\mask_sentinel_TOA_S2_argemiro_20190202.tif, location: argemiro
Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190910.tif and mask: D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\2021\mask_sentinel_TOA_S2_argemiro_20210517.tif, location: argemiro
Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20191124.tif and mask: D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\2025\mask_sentinel_TOA_S2_argemiro_20250531.tif, location: argemiro
Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2020\sentinel_BOA_S2_SR_argemiro_20200323.tif and mask: D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\2022\mask_sentinel_TOA_S2_argemiro_20220924.tif, location: argemiro
Processing image: D:/GeoPipe/data/02_boa_images/argemiro\2020\sentinel_BOA_S2_SR_argemiro_20

## Clean images

In [ ]:
locations = ['argemiro', 'engenheiro_avidos', 'gramame', 'lagoa_do_arroz', 'mares', 'sume']
for location in locations:
    cloud_removal(
        path_images = temp_image_dir.replace("all", ""),
        path_masks = temp_mask_dir.replace("all", "").replace("fmask", ""),
        output_path = temp_clean_image_dir.replace("all", ""),
        location_name = location,
        cloud_and_cloud_shadow_pixels = [1, 2],
        init_date = init_date_formatted,
        final_date = final_date_formatted,
        skip_clean = False,
        color_file_log_path = temp_color_log_dir.replace("all", ""),
        cloud_mask_algorithm = "fmask",
        reconstruction_algorithm = "temporal_interpolation",
        max_workers = 4,
        time_series_images_path="D:/GeoPipe/data/02_boa_images",
        time_series_masks_path="D:/GeoPipe/data/03_cloud_masks"
    )

Cleaning Images:   0%|          | 0/236 [00:00<?, ?file/s]No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170921.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170921.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20171011.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170728.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170728.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20171016.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170723.tif'. Skipping band update.
No corresponding 6-band image found for 'mask_sentinel_TOA_S2_engenheiro_avidos_20170723.tif'. Skipping band update.
No cor

: 